# DPO-SDXL × DrawBench — Standard Sampling (Colab)

Doing the same for standard sampling, but using the Diffision-DPO model (SDXL, Direct Preference Optimization)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import gc
import json
import os
import shutil
import tempfile
import time
from pathlib import Path

import torch
from diffusers import DDIMScheduler, StableDiffusionXLPipeline, UNet2DConditionModel
from IPython.display import display
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"torch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Config

In [ ]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DPO_UNET_ID = "mhdang/dpo-sdxl-text2image-v1"
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 5.5
HEIGHT = 1024
WIDTH = 1024
SEEDS = [2026] # [42, 123, 2026]

OUTPUT_DIR = Path("/content/outputs/dpo_sdxl_drawbench")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling/dpo_sdxl_drawbench")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY = 25

# How many prompts to run. Set to None for all 200.
NUM_PROMPTS = None
PROMPT_SEED = 42

## DrawBench Prompts (200)

In [ ]:
import random
import pandas as pd

CSV_URL = "https://huggingface.co/datasets/sayakpaul/drawbench/raw/main/DrawBench%20Prompts%20-%20Sheet1.csv"
df = pd.read_csv(CSV_URL)
DRAWBENCH_PROMPTS = list(enumerate(df["Prompts"].tolist()))

if NUM_PROMPTS is not None and NUM_PROMPTS < len(DRAWBENCH_PROMPTS):
    rng = random.Random(PROMPT_SEED)
    selected_prompts = sorted(rng.sample(DRAWBENCH_PROMPTS, NUM_PROMPTS))
    print(f"Randomly selected {NUM_PROMPTS}/{len(DRAWBENCH_PROMPTS)} prompts (seed={PROMPT_SEED})")
else:
    selected_prompts = DRAWBENCH_PROMPTS
    print(f"Running all {len(selected_prompts)} prompts")

print(f"First 5: {[f'{i}: {p[:40]}...' for i, p in selected_prompts[:5]]}")
df.head()

In [ ]:
# select last 100 prompts
selected_prompts = selected_prompts[-100:]


## Load Pipeline (DPO UNet)

In [ ]:
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, use_safetensors=True
)

# Replace UNet with DPO-finetuned version
unet = UNet2DConditionModel.from_pretrained(
    DPO_UNET_ID, subfolder="unet", torch_dtype=DTYPE
)
pipe.unet = unet

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)
if DEVICE == "cuda":
    pipe.enable_attention_slicing()
    pipe.enable_vae_tiling()

print(f"Pipeline loaded with DPO UNet from {DPO_UNET_ID}")

## Generate

In [ ]:
def checkpoint_to_drive(files, tag):
    """Zip only the given files and copy to Drive with flush."""
    with tempfile.TemporaryDirectory() as tmp:
        for f in files:
            shutil.copy2(f, Path(tmp) / f.name)
        zp = shutil.make_archive(f"/content/{tag}", "zip", tmp)
    dest = DRIVE_DIR / f"{tag}.zip"
    shutil.copy2(zp, dest)
    os.sync()
    size_mb = Path(zp).stat().st_size / 1e6
    print(f"  >> Checkpoint saved: {dest.name} ({size_mb:.1f} MB, {len(files)} images)")

total = len(SEEDS) * len(selected_prompts)
done = 0

for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    manifest = seed_dir / "manifest.jsonl"
    pending_files = []
    ckpt_count = 5

    for prompt_idx, prompt in selected_prompts:
        t0 = time.time()
        generator = torch.Generator(device=DEVICE).manual_seed(seed)
        result = pipe(
            prompt,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            height=HEIGHT,
            width=WIDTH,
            generator=generator,
        )
        img = result.images[0]
        fname = f"dpo_sdxl_prompt{prompt_idx:03d}_seed{seed}.png"
        path = seed_dir / fname
        img.save(path)
        pending_files.append(path)
        del result, img
        torch.cuda.empty_cache()

        with open(manifest, "a") as f:
            f.write(json.dumps({"prompt_id": prompt_idx, "prompt": prompt, "seed": seed, "file": fname}) + "\n")

        done += 1
        elapsed = time.time() - t0
        print(f"[{done}/{total}] seed={seed} prompt={prompt_idx} ({elapsed:.1f}s)")

        if len(pending_files) == CHECKPOINT_EVERY:
            ckpt_count += 1
            checkpoint_to_drive(pending_files, f"dpo_sdxl_drawbench_ckpt{ckpt_count}_seed{seed}")
            pending_files = []

    if pending_files:
        ckpt_count += 1
        checkpoint_to_drive(pending_files, f"dpo_sdxl_drawbench_ckpt{ckpt_count}_seed{seed}")
        pending_files = []

print(f"\nDone. {done} images saved to {OUTPUT_DIR}")

## Preview

In [ ]:
sample_paths = sorted(OUTPUT_DIR.rglob("*.png"))[-4:]
for p in sample_paths:
    print(p.name)
    display(Image.open(p))

## Copy to Google Drive

In [ ]:
for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f"seed{seed}"
    if not seed_dir.exists():
        print(f"Skipping seed {seed} — no output dir")
        continue
    n = len(list(seed_dir.glob("*.png")))
    tag = f"dpo_sdxl_drawbench_{n}imgs_seed{seed}_final"
    zp = shutil.make_archive(f"/content/{tag}", "zip", seed_dir)
    dest = DRIVE_DIR / f"{tag}.zip"
    shutil.copy2(zp, dest)
    print(f"{dest.name}: {Path(zp).stat().st_size / 1e6:.1f} MB")
os.sync()
print("All final zips flushed to Drive.")